# Task 1 — Data Exploration and Enrichment

**Objective:** understand the starter unified-schema dataset, explore its contents, and enrich it
with additional sourced observations, events, and impact links useful for forecasting
**Access** (`ACC_OWNERSHIP`) and **Usage** (`USG_DIGITAL_PAYMENT`).

The unified schema uses one column set for all rows; `record_type` determines interpretation:
`observation` (measured values), `event` (policies/launches/milestones — *no pillar pre-assigned*),
`impact_link` (event → indicator relationships via `parent_id`), and `target` (official goals).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.data_loader import RAW_DIR
from src.schema import validate_records

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

RAW_XLSX = RAW_DIR / "ethiopia_fi_unified_data.xlsx"
main = pd.read_excel(RAW_XLSX, sheet_name="ethiopia_fi_unified_data")
impact = pd.read_excel(RAW_XLSX, sheet_name="Impact_sheet")
ref = pd.read_excel(RAW_DIR / "reference_codes.xlsx")
print(f"main sheet: {main.shape}, impact sheet: {impact.shape}, reference codes: {ref.shape}")

main sheet: (43, 34), impact sheet: (14, 35), reference codes: (71, 4)


## 1. Schema overview

All rows share 34 columns; the impact sheet adds `parent_id`.

In [2]:
print("Main sheet columns:")
print(main.columns.tolist())
main.head(3)

Main sheet columns:
['record_id', 'record_type', 'category', 'pillar', 'indicator', 'indicator_code', 'indicator_direction', 'value_numeric', 'value_text', 'value_type', 'unit', 'observation_date', 'period_start', 'period_end', 'fiscal_year', 'gender', 'location', 'region', 'source_name', 'source_type', 'source_url', 'confidence', 'related_indicator', 'relationship_type', 'impact_direction', 'impact_magnitude', 'impact_estimate', 'lag_months', 'evidence_basis', 'comparable_country', 'collected_by', 'collection_date', 'original_text', 'notes']


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,%,2014-12-31,NaT,NaT,2014,all,national,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,%,2017-12-31,NaT,NaT,2017,all,national,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,all,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN


## 2. Record counts by type, pillar, source, and confidence

In [3]:
display(main["record_type"].value_counts().to_frame("count"))
display(main.groupby(["record_type", "pillar"], dropna=False).size().to_frame("count"))
display(main["source_type"].value_counts(dropna=False).to_frame("count"))
display(main["confidence"].value_counts(dropna=False).to_frame("count"))

,count
record_type,
observation,30
event,10
target,3


count
record_type pillar              
event       NaN               10
observation ACCESS            14
            AFFORDABILITY      1
            GENDER             4
            USAGE             11
target      ACCESS             2
            GENDER             1

,count
source_type,
operator,15
survey,10
regulator,7
research,4
policy,3
calculated,2
news,2


,count
confidence,
high,40
medium,3


## 3. Temporal range and indicator coverage

In [4]:
obs = main[main.record_type == "observation"].copy()
obs["observation_date"] = pd.to_datetime(obs["observation_date"])
print(f"Observations span {obs.observation_date.min().date()} to {obs.observation_date.max().date()}")

coverage = (obs.assign(year=obs.observation_date.dt.year)
              .groupby(["indicator_code", "pillar"])
              .agg(n_obs=("record_id", "count"),
                   years=("year", lambda y: sorted(set(y)))))
coverage.sort_values("n_obs", ascending=False)

Observations span 2014-12-31 to 2025-12-31


,,n_obs,years
indicator_code,pillar,,
ACC_OWNERSHIP,ACCESS,6,"[2014, 2017, 2021, 2024]"
ACC_FAYDA,ACCESS,3,"[2024, 2025]"
ACC_4G_COV,ACCESS,2,"[2023, 2025]"
ACC_MM_ACCOUNT,ACCESS,2,"[2021, 2024]"
GEN_GAP_ACC,GENDER,2,"[2021, 2024]"
USG_P2P_COUNT,USAGE,2,"[2024, 2025]"
ACC_MOBILE_PEN,ACCESS,1,[2025]
GEN_GAP_MOBILE,GENDER,1,[2024]
GEN_MM_SHARE,GENDER,1,[2024]


**Key gap:** `USG_DIGITAL_PAYMENT` — one of the two forecast targets — has **no
observations at all** in the starter data. `ACC_OWNERSHIP` is missing its 2011 point (14%),
and mobile money has no pre-2021 history. These drive the enrichment below.

## 4. Events and impact links

In [5]:
events = main[main.record_type == "event"].copy()
events["observation_date"] = pd.to_datetime(events["observation_date"])
events[["record_id", "category", "indicator", "observation_date", "source_name", "confidence"]] \
    .sort_values("observation_date")

,record_id,category,indicator,observation_date,source_name,confidence
33,EVT_0001,product_launch,Telebirr Launch,2021-05-17,Ethio Telecom,high
41,EVT_0009,policy,NFIS-II Strategy Launch,2021-09-01,NBE,high
34,EVT_0002,market_entry,Safaricom Ethiopia Commercial Launch,2022-08-01,News,high
35,EVT_0003,product_launch,M-Pesa Ethiopia Launch,2023-08-01,Safaricom,high
36,EVT_0004,infrastructure,Fayda Digital ID Program Rollout,2024-01-01,NIDP,high
37,EVT_0005,policy,Foreign Exchange Liberalization,2024-07-29,NBE,high
38,EVT_0006,milestone,P2P Transaction Count Surpasses ATM,2024-10-01,EthSwitch,high
39,EVT_0007,partnership,M-Pesa EthSwitch Integration,2025-10-27,EthSwitch,high
42,EVT_0010,pricing,Safaricom Ethiopia Price Increase,2025-12-15,News,high
40,EVT_0008,infrastructure,EthioPay Instant Payment System Launch,2025-12-18,NBE/EthSwitch,high


In [6]:
links = impact.merge(
    events[["record_id", "indicator", "category", "observation_date"]].rename(
        columns={"record_id": "parent_id", "indicator": "event_name",
                 "category": "event_category", "observation_date": "event_date"}),
    on="parent_id", how="left")
links[["record_id", "event_name", "pillar", "related_indicator", "impact_direction",
       "impact_magnitude", "impact_estimate", "lag_months", "evidence_basis",
       "comparable_country"]]

,record_id,event_name,pillar,related_indicator,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country
0,IMP_0001,Telebirr Launch,ACCESS,ACC_OWNERSHIP,increase,high,15.0,12,literature,Kenya
1,IMP_0002,Telebirr Launch,USAGE,USG_TELEBIRR_USERS,increase,high,NaN,3,empirical,NaN
2,IMP_0003,Telebirr Launch,USAGE,USG_P2P_COUNT,increase,high,25.0,6,empirical,NaN
3,IMP_0004,Safaricom Ethiopia Commercial Launch,ACCESS,ACC_4G_COV,increase,medium,15.0,12,empirical,NaN
4,IMP_0005,Safaricom Ethiopia Commercial Launch,AFFORDABILITY,AFF_DATA_INCOME,decrease,medium,-20.0,12,literature,Rwanda
5,IMP_0006,M-Pesa Ethiopia Launch,USAGE,USG_MPESA_USERS,increase,high,NaN,3,empirical,NaN
6,IMP_0007,M-Pesa Ethiopia Launch,ACCESS,ACC_MM_ACCOUNT,increase,medium,5.0,6,theoretical,NaN
7,IMP_0008,Fayda Digital ID Program Rollout,ACCESS,ACC_OWNERSHIP,increase,medium,10.0,24,literature,India
8,IMP_0009,Fayda Digital ID Program Rollout,GENDER,GEN_GAP_ACC,decrease,medium,-5.0,24,literature,India
9,IMP_0010,Foreign Exchange Liberalization,AFFORDABILITY,AFF_DATA_INCOME,increase,high,30.0,3,empirical,NaN


**Observations on the starter impact links:**
- Telebirr (EVT_0001) links to `ACC_OWNERSHIP`, `USG_TELEBIRR_USERS`, `USG_P2P_COUNT` — but *not* to
  `ACC_MM_ACCOUNT` or `USG_DIGITAL_PAYMENT`, despite being the dominant driver of both.
- NFIS-II (EVT_0009) and the P2P>ATM milestone (EVT_0006) have **no links at all**.
- Events with no pillar assignment is deliberate (see `data/raw/README.md`): pillar lives on the
  impact link, keeping event records unbiased.

## 5. Enrichment

New records are defined in [`src/enrichment.py`](../src/enrichment.py) with full source
documentation, and written to `data/processed/`. Additions (see
[`reports/data_enrichment_log.md`](../reports/data_enrichment_log.md) for the full log):

- **13 observations** — the missing 2011 Findex baseline; `USG_DIGITAL_PAYMENT` for 2017/2021/2024
  (the Usage forecast target); early mobile-money points (2014, 2017); the Telebirr adoption ramp
  (2022, 2023, 2024); an early M-Pesa point; wage-payment usage; derived male/female 2024 ownership.
- **2 events** — NBE Payment Instrument Issuers Directive ONPS/01/2020 (the regulation that enabled
  non-bank mobile money) and the 2022 mandatory digital fuel-payment rule (a forced-adoption use case).
- **9 impact links** — connecting Telebirr to `ACC_MM_ACCOUNT` and `USG_DIGITAL_PAYMENT`, the new
  events to their affected indicators, and filling gaps (NFIS-II, M-Pesa, EthioPay, Fayda→eKYC).

In [7]:
from src.enrichment import NEW_EVENTS, NEW_IMPACT_LINKS, NEW_OBSERVATIONS, build_enriched

main_enr, impact_enr = build_enriched(write=True)
print(f"Enriched main sheet: {len(main)} -> {len(main_enr)} records")
print(f"Enriched impact sheet: {len(impact)} -> {len(impact_enr)} records")
main_enr["record_type"].value_counts().to_frame("count")

Enriched main sheet: 43 -> 58 records
Enriched impact sheet: 14 -> 23 records


,count
record_type,
observation,43
event,12
target,3


In [8]:
added = main_enr[main_enr.collected_by == "KalkidanAsfaw"]
added[["record_id", "record_type", "pillar", "indicator", "indicator_code",
       "value_numeric", "observation_date", "source_name", "confidence"]]

,record_id,record_type,pillar,indicator,indicator_code,value_numeric,observation_date,source_name,confidence
43,REC_0034,observation,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,14.00,2011-12-31,Global Findex 2011,high
44,REC_0035,observation,ACCESS,Mobile Money Account Rate,ACC_MM_ACCOUNT,0.03,2014-12-31,Global Findex 2014,medium
45,REC_0036,observation,ACCESS,Mobile Money Account Rate,ACC_MM_ACCOUNT,0.30,2017-12-31,Global Findex 2017,medium
46,REC_0037,observation,USAGE,Made or Received Digital Payment,USG_DIGITAL_PAYMENT,11.90,2017-12-31,Global Findex 2017,medium
47,REC_0038,observation,USAGE,Made or Received Digital Payment,USG_DIGITAL_PAYMENT,20.00,2021-12-31,Global Findex 2021,medium
48,REC_0039,observation,USAGE,Made or Received Digital Payment,USG_DIGITAL_PAYMENT,35.00,2024-11-29,Global Findex 2025,medium
49,REC_0040,observation,USAGE,Received Wages Into Account,USG_WAGES,15.00,2024-11-29,Global Findex 2025,medium
50,REC_0041,observation,USAGE,Telebirr Registered Users,USG_TELEBIRR_USERS,20000000.00,2022-06-30,Ethio Telecom 2021/22 Report,medium
51,REC_0042,observation,USAGE,Telebirr Registered Users,USG_TELEBIRR_USERS,34300000.00,2023-06-30,Ethio Telecom 2022/23 Report,high
52,REC_0043,observation,USAGE,Telebirr Registered Users,USG_TELEBIRR_USERS,47500000.00,2024-06-30,Ethio Telecom 2023/24 Report,high


## 6. Schema validation of the enriched dataset

In [9]:
problems = validate_records(main_enr)
print("Schema problems:", problems if problems else "none")

event_ids = set(main_enr.loc[main_enr.record_type == "event", "record_id"])
orphans = set(impact_enr["parent_id"]) - event_ids
print("Orphan impact links:", orphans if orphans else "none")

cov = (main_enr[main_enr.record_type == "observation"]
       .assign(year=pd.to_datetime(main_enr.observation_date).dt.year)
       .groupby("indicator_code")["year"].agg(["count", "min", "max"]))
cov.sort_values("count", ascending=False)

Schema problems: none
Orphan impact links: none


,count,min,max
indicator_code,,,
ACC_OWNERSHIP,9,2011,2024
ACC_MM_ACCOUNT,4,2014,2024
USG_TELEBIRR_USERS,4,2022,2025
USG_DIGITAL_PAYMENT,3,2017,2024
ACC_FAYDA,3,2024,2025
USG_MPESA_USERS,2,2024,2024
ACC_4G_COV,2,2023,2025
GEN_GAP_ACC,2,2021,2024
USG_P2P_COUNT,2,2024,2025


## Summary

- All three starter files load successfully; the unified schema is internally consistent
  (events pillar-free, impact links resolve to parent events).
- The starter data was **missing the Usage forecast target entirely** — now covered with three
  Findex survey points (2017: ~12%, 2021: ~20%, 2024: ~35%).
- The Telebirr adoption ramp (2021 launch → 54.8M users in 2025) is now observable year by year,
  which Task 3 uses to calibrate event-effect shapes.
- Two regulatory events were added that materially explain the timing of mobile money growth:
  the 2020 PSP directive (market opener) and the 2022 fuel-payment mandate (forced adoption).
- Enriched outputs: `data/processed/ethiopia_fi_unified_data_enriched.csv` and
  `data/processed/impact_links_enriched.csv`.